# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (BF16)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out no cells below after first installation. Colab has fresh env everytime. UV parallelizes installations so should be quick.

In [17]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!wget -qO- https://astral.sh/uv/install.sh | sh

# Make uv findable in subsequent cells
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

# Verify
!which uv && uv --version

downloading uv 0.11.8 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/usr/local/bin/uv
uv 0.11.8 (x86_64-unknown-linux-gnu)


In [19]:
constraints = """torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57
"""

with open("/content/constraints.txt", "w") as f:
    f.write(constraints)

!cat /content/constraints.txt

torch==2.5.1
torchvision==0.20.1
torchaudio==2.5.1
transformers>=4.48,<=4.57


In [20]:
!uv pip install --system torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 \
    --index-url https://download.pytorch.org/whl/cu121

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 103ms


In [21]:
import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("CUDA version torch was built with:", torch.version.cuda)

torch version: 2.5.1+cu121
CUDA available: True
Device count: 1
Device name: NVIDIA A100-SXM4-80GB
CUDA version torch was built with: 12.1


In [22]:
# removed bitsandbytes package cause not using it anymore.
!uv pip install --system \
    sympy numpy transformers vllm tqdm \
    antlr4-python3-runtime==4.11.1 accelerate \
    -c /content/constraints.txt

Using Python 3.12.13 environment at: /usr
Checked 7 packages in 102ms


In [23]:
# If vllm import fails the first time after install, restart the session and re-run all cells. Same for any persistent setup errors.
import torch
import vllm
print("Post-restart checks:")
print("  torch:", torch.__version__)
print("  CUDA available:", torch.cuda.is_available())
print("  Device:", torch.cuda.get_device_name(0))
print("  vllm:", vllm.__version__)
print("  GPU memory free:", round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), "GB")

Post-restart checks:
  torch: 2.5.1+cu121
  CUDA available: True
  Device: NVIDIA A100-SXM4-80GB
  vllm: 0.7.3
  GPU memory free: 9.3 GB


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_DIR` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [24]:
from pathlib import Path
import sys
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/cse151b")
GIVEN_DATA_DIR = PROJECT_DIR / "given_data"
OUTPUT_DIR = PROJECT_DIR  # final outputs land here on Drive

# Local hot-path directory (Colab disk — fast, reliable, but wiped on runtime death)
LOCAL_DIR = Path("/content/local_results")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Inputs
# DATA_PATH = str(GIVEN_DATA_DIR / "public.jsonl")
DATA_PATH = str(GIVEN_DATA_DIR / "private.jsonl")

# Hot-path outputs — written to local disk during generation
RESPONSES_PATH = LOCAL_DIR / "responses.jsonl"
LOG_PATH = LOCAL_DIR / "responses.log"

# Drive backup of responses (snapshot target during run, restore source after restart)
RESPONSES_BACKUP = OUTPUT_DIR / "responses.jsonl"

# Final outputs — written to Drive (only one write each, at the end of their step)
SCORED_PATH = OUTPUT_DIR / "scored_results.jsonl"
SUBMISSION_PATH = OUTPUT_DIR / "submission.jsonl"

# Make given_data importable
sys.path.insert(0, str(GIVEN_DATA_DIR))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert OUTPUT_DIR.exists(), f"Drive not mounted? {OUTPUT_DIR} missing"

# If a Drive backup exists from a previous session but no local copy, restore it
if RESPONSES_BACKUP.exists() and not RESPONSES_PATH.exists():
    shutil.copy2(RESPONSES_BACKUP, RESPONSES_PATH)
    print(f"Restored {RESPONSES_PATH.stat().st_size} bytes from Drive backup")

print(f"Project: {PROJECT_DIR}")
print(f"Inputs:  {GIVEN_DATA_DIR}")
print(f"Hot dir: {LOCAL_DIR}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Existing local responses: {RESPONSES_PATH.exists()}")
print(f"Existing Drive backup:    {RESPONSES_BACKUP.exists()}")

Project: /content/drive/MyDrive/cse151b
Inputs:  /content/drive/MyDrive/cse151b/given_data
Hot dir: /content/local_results
Outputs: /content/drive/MyDrive/cse151b
Existing local responses: True
Existing Drive backup:    False


In [1]:
import json
import re
import time
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
MAX_TOKENS  = 32768 # was 32768. 38912 recommended

# Set GPU_ID earlier if training off of colab and have multiple GPUs. Need to declare before torch is imported.
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

NameError: name 'os' is not defined

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [26]:
with open(DATA_PATH) as f:
    data = [json.loads(line) for line in f]

In [27]:
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 943 questions  (300 MCQ, 643 free-form)


In [28]:
# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))


── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [29]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "If the problem has multiple sub-answers, place them inside a single \\boxed{} "
    "separated by commas, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices, then select the single best option. "
    "Please reason step by step, and put only the letter of your chosen option "
    "within \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [30]:
# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with vLLM

We load Qwen3-4B-Thinking-2507 in BF16 for maximum throughput on A100. Quantization (BnB INT8) is said to be slower (needs verification) than BF16 on A100 due to dequantization overhead exceeding memory bandwidth savings. AWQ would be the right choice if memory becomes tight.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [31]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",                   # drop bnb on A100 — BF16 is faster AND better quality
    trust_remote_code=True,
    max_model_len=24576,
    gpu_memory_utilization=0.90,
    max_num_batched_tokens=24576,       # match max_model_len; 32768 was overprovisioned
    max_num_seqs=128,                   # 256 is fine if you have headroom, 128 is safer. Tests set OOM on 128.
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
    disable_log_stats=True,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,                   # tame the long-tail thinking traces
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed = 5,
    stop=None,
)

print("Model loaded.")

INFO 05-02 05:53:40 config.py:549] This model supports multiple tasks: {'classify', 'embed', 'reward', 'score', 'generate'}. Defaulting to 'generate'.
INFO 05-02 05:53:40 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=16384, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=Qwen/Qwen3-4B-Thinking-2507, n

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-02 05:53:45 model_runner.py:1115] Loading model weights took 7.5454 GB


OutOfMemoryError: CUDA out of memory. Tried to allocate 304.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 22.81 MiB is free. Including non-PyTorch memory, this process has 79.21 GiB memory in use. Of the allocated memory 78.40 GiB is allocated by PyTorch, and 320.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import utils
from utils import last_boxed_only_string, remove_boxed
from judger import Judger

In [ ]:
def extract_letter(text: str) -> str:
    # Strip thinking trace — only look at content after </think>
    think_end = text.rfind("</think>")
    search_text = text[think_end + len("</think>"):] if think_end >= 0 else text

    # First try: pull the last \boxed{...} content using utils' brace-aware parser
    boxed = last_boxed_only_string(search_text)
    if boxed is not None:
        inner = remove_boxed(boxed)
        if inner:
            m = re.search(r"[A-Za-z]", inner)
            if m:
                return m.group(0).upper()
    # Fallback: last standalone capital letter in the post-think response
    matches = re.findall(r"\b([A-Z])\b", search_text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

In [ ]:
judger = Judger(strict_extract=False)
print("Scoring helpers ready.")

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:
CHUNK_SIZE = 250
completed_ids = set()

In [ ]:
# ── Resume: skip IDs we already have responses for ────────────────────────
if RESPONSES_PATH.exists():
    with open(RESPONSES_PATH) as f:
        for line in f:
            try:
                completed_ids.add(json.loads(line)["id"])
            except (json.JSONDecodeError, KeyError):
                continue

In [ ]:
remaining = [d for d in data if d.get("id") not in completed_ids]
print(f"Resuming generation: {len(completed_ids)} done, {len(remaining)} to go")

In [ ]:
def make_prompt(item):
    system, user = build_prompt(item["question"], item.get("options"))
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False, add_generation_prompt=True,
    )

In [ ]:
t_start = time.time()
pbar = tqdm(total=len(remaining), desc="Generating", unit="q")

with open(RESPONSES_PATH, "a") as fout, open(LOG_PATH, "a") as flog:
    flog.write(f"\n=== Generation started {time.strftime('%Y-%m-%d %H:%M:%S')} ===\n")
    flog.flush()

    for i in range(0, len(remaining), CHUNK_SIZE):
        chunk = remaining[i : i + CHUNK_SIZE]
        prompts = [make_prompt(item) for item in chunk]
        t_chunk = time.time()

        try:
            outputs = llm.generate(prompts, sampling_params=sampling_params, use_tqdm=False)
        except Exception as e:
            flog.write(f"CHUNK FAIL items {i}-{i+len(chunk)}: {e}\n")
            flog.flush()
            # Fall back to one-by-one to isolate the bad item
            outputs = []
            for p, item in zip(prompts, chunk):
                try:
                    outputs.extend(llm.generate([p], sampling_params=sampling_params, use_tqdm=False))
                except Exception as e2:
                    flog.write(f"  ITEM FAIL id={item.get('id')}: {e2}\n")
                    outputs.append(None)
            flog.flush()

        for item, out in zip(chunk, outputs):
            if out is None:
                record = {
                    "id": item.get("id"),
                    "is_mcq": bool(item.get("options")),
                    "gold": item.get("answer"),
                    "response": "",
                    "error": "generation_failed",
                }
            else:
                record = {
                    "id": item.get("id"),
                    "is_mcq": bool(item.get("options")),
                    "gold": item.get("answer"),
                    "response": out.outputs[0].text.strip(),
                }
            fout.write(json.dumps(record) + "\n")
            pbar.update(1)

        fout.flush(); os.fsync(fout.fileno())

        chunk_time = time.time() - t_chunk
        elapsed = time.time() - t_start
        msg = (f"Chunk {i//CHUNK_SIZE+1}: {len(chunk)} items in {chunk_time:.1f}s "
               f"| elapsed: {elapsed/60:.1f}min")
        flog.write(msg + "\n"); flog.flush()

        # Snapshot to Drive every 5 chunks (~250 items)
        chunk_idx = i // CHUNK_SIZE
        if chunk_idx > 0 and chunk_idx % 5 == 0:
            try:
                import shutil
                shutil.copy2(RESPONSES_PATH, RESPONSES_BACKUP)
                flog.write(f"  Drive backup: {RESPONSES_BACKUP}\n"); flog.flush()
            except Exception as e:
                flog.write(f"  Drive backup FAILED: {e}\n"); flog.flush()
                # Don't crash the run — local file is the source of truth

pbar.close()

# Final copy to Drive so subsequent steps (and the next session) have it
import shutil
try:
    shutil.copy2(RESPONSES_PATH, RESPONSES_BACKUP)
    print(f"\nGeneration complete.")
    print(f"  Local: {RESPONSES_PATH}")
    print(f"  Drive: {RESPONSES_BACKUP}")
except Exception as e:
    print(f"\nGeneration complete locally but Drive copy failed: {e}")
    print(f"  Manually copy {RESPONSES_PATH} to Drive before relying on it.")

In [ ]:
print(f"pbar.n: {pbar.n} / {pbar.total}")
print(f"len(remaining): {len(remaining)}")
print(f"len(completed_ids): {len(completed_ids)}")
with open(RESPONSES_PATH) as f:
    lines = f.readlines()
print(f"Records in file: {len(lines)}")

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
# Load responses from Step 6
records = []
with open(RESPONSES_PATH) as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

print(f"Loaded {len(records)} responses to score")

In [ ]:
results = []

In [ ]:
import signal

class ScoringTimeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise ScoringTimeout("scoring timeout")

# Register handler once, outside the loop
signal.signal(signal.SIGALRM, _timeout_handler)

In [ ]:
for r in tqdm(records, desc="Scoring"):
    # Carry forward generation failures without crashing the scorer
    if r.get("error") == "generation_failed":
        r["correct"] = False
        results.append(r)
        continue

    is_mcq = r["is_mcq"]
    gold = r["gold"]
    response = r["response"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        signal.alarm(120)  # 120 second per-item ceiling
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False
        finally:
            signal.alarm(0)

    r["correct"] = correct
    results.append(r)

In [ ]:
# Save the scored file
with open(SCORED_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

print(f"Scoring complete. {len(results)} results written to {SCORED_PATH}")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
with open(SCORED_PATH) as f:
    results = [json.loads(line) for line in f]

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

In [ ]:
def acc(subset):
    return sum(r.get("correct", False) for r in subset) / len(subset) * 100 if subset else 0.0

In [ ]:
print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r.get('correct', False) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r.get('correct', False) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r.get('correct', False) for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # False when running on the private test set

In [ ]:
with open(SCORED_PATH) as fin, open(SUBMISSION_PATH, "w") as fout:
    for line in fin:
        r = json.loads(line)
        if r.get("error") == "generation_failed":
            continue  # skip failed entries from submission
        if SAVE_EVAL:
            record = {k: r[k] for k in ["id", "is_mcq", "gold", "response", "correct"] if k in r}
        else:
            record = {k: r[k] for k in ["id", "is_mcq", "response"]}
        fout.write(json.dumps(record) + "\n")

print(f"Wrote submission to {SUBMISSION_PATH}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!